# 6. Advanced Recommendation System

## Objective

Improve the existing tourism recommendation system by building a
stronger hybrid recommendation approach.

### Approach

The advanced recommender builds on the previous recommendation system
by combining:

- Collaborative Filtering
- Content-Based Filtering
- User behavioral information
- Attraction popularity

The goal is to generate more personalized and robust tourism
recommendations while avoiding attractions already visited by the user.

# 6. Advanced Recommendation System

## Objective

Build an improved hybrid recommendation system by extending the
recommendation approach developed in Notebook 4.

## Approach

The advanced recommender combines multiple signals to improve
personalization and recommendation quality:

- Collaborative Filtering
- Content-Based Filtering
- Attraction popularity
- User behavior

The system will generate personalized recommendations while avoiding
attractions that the user has already visited.

## Key Goals

- Improve recommendation relevance
- Handle users with limited interaction history
- Use information from the full attraction catalog
- Provide recommendations that are personalized and practical

In [34]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully.")

Libraries imported successfully.


In [35]:
df = pd.read_csv("tourism_cleaned_engineered.csv")

kmeans_model = joblib.load("kmeans_clustering_model.pkl")
clustering_scaler = joblib.load("clustering_scaler.pkl")
clustering_features = joblib.load("clustering_feature_columns.pkl")
cluster_names = joblib.load("cluster_names.pkl")

print("Dataset Shape:", df.shape)
print("Clustering Features:", clustering_features)
print("Cluster Names:", cluster_names)

Dataset Shape: (52930, 24)
Clustering Features: ['total_visits', 'unique_attractions', 'avg_rating', 'unique_visit_modes']
Cluster Names: {0: 'Occasional High-Rating Visitors', 1: 'Multi-Mode Travelers', 2: 'Low-Satisfaction Visitors', 3: 'Frequent & Diverse Travelers', 4: 'Repeat Visitors'}


In [36]:
user_features = df.groupby("UserId").agg(
    total_visits=("TransactionId", "count"),
    unique_attractions=("AttractionId", "nunique"),
    avg_rating=("Rating", "mean"),
    unique_visit_modes=("VisitModeId", "nunique")
).reset_index()

X_user = user_features[clustering_features]

X_user_scaled = clustering_scaler.transform(X_user)

user_features["Cluster"] = kmeans_model.predict(X_user_scaled)

user_features["Segment"] = user_features["Cluster"].map(cluster_names)

print("User segmentation completed.")
print("\nSegment Distribution:")
print(user_features["Segment"].value_counts())

User segmentation completed.

Segment Distribution:
Segment
Occasional High-Rating Visitors    19760
Repeat Visitors                     5372
Low-Satisfaction Visitors           5295
Multi-Mode Travelers                2527
Frequent & Diverse Travelers         576
Name: count, dtype: int64


In [37]:
attraction_popularity = df.groupby("AttractionId").agg(
    visit_count=("TransactionId", "count"),
    avg_rating=("Rating", "mean"),
    unique_users=("UserId", "nunique")
).reset_index()

# Normalize popularity signals
popularity_scaler = StandardScaler()

attraction_popularity[
    ["visit_count_scaled", "avg_rating_scaled", "unique_users_scaled"]
] = popularity_scaler.fit_transform(
    attraction_popularity[
        ["visit_count", "avg_rating", "unique_users"]
    ]
)

# Combined popularity score
attraction_popularity["popularity_score"] = (
    0.5 * attraction_popularity["visit_count_scaled"] +
    0.3 * attraction_popularity["unique_users_scaled"] +
    0.2 * attraction_popularity["avg_rating_scaled"]
)

print("Attraction popularity calculated.")
print("\nTop 10 Popular Attractions:")
print(
    attraction_popularity
    .sort_values("popularity_score", ascending=False)
    [["AttractionId", "visit_count", "unique_users", "avg_rating", "popularity_score"]]
    .head(10)
)

Attraction popularity calculated.

Top 10 Popular Attractions:
    AttractionId  visit_count  unique_users  avg_rating  popularity_score
2            640        13198         11487    4.267086          3.438255
9            841         6429          5605    4.646601          1.710347
6            748         5815          5415    4.157524          1.261456
8            824         3359          3183    4.219411          0.573977
5            737         3352          3156    4.194809          0.553632
3            650         3044          2808    3.976347          0.314417
1            481         2104          1992    4.275665          0.232190
4            673         2914          2754    3.800618          0.171541
7            749         2190          2152    3.895434          0.024735
23          1171         2235           855    4.079195         -0.009763


In [38]:
user_item_matrix = df.pivot_table(
    index="UserId",
    columns="AttractionId",
    values="Rating",
    aggfunc="mean",
    fill_value=0
)

item_user_matrix = user_item_matrix.T

collaborative_similarity = cosine_similarity(item_user_matrix)

collaborative_similarity_df = pd.DataFrame(
    collaborative_similarity,
    index=item_user_matrix.index,
    columns=item_user_matrix.index
)

print("Collaborative filtering structure created.")
print("User-Item Matrix Shape:", user_item_matrix.shape)
print("Item-Item Similarity Shape:", collaborative_similarity_df.shape)

Collaborative filtering structure created.
User-Item Matrix Shape: (33530, 30)
Item-Item Similarity Shape: (30, 30)


In [39]:
# Load attraction type lookup exactly as used in Notebook 4

type_lookup = pd.read_excel('Tourism_Analytics/data/Type.xlsx')

type_lookup_map = type_lookup.set_index(
    "AttractionTypeId"
)["AttractionType"].to_dict()


def normalize_type(val):
    val_str = str(val).strip()

    if val_str.isdigit():
        return type_lookup_map.get(
            int(val_str),
            "Other"
        )

    return val_str


item_full["AttractionType_clean"] = item_full[
    "AttractionTypeId"
].apply(normalize_type)

print("Attraction type normalization completed.")

print("\nAttraction Type Distribution:")
print(
    item_full["AttractionType_clean"]
    .value_counts()
    .head(20)
)

print(
    "\nMissing/Other types:",
    (item_full["AttractionType_clean"] == "Other").sum()
)

Attraction type normalization completed.

Attraction Type Distribution:
AttractionType_clean
Museum                            367
Temple                            340
Market                            325
Beach                             318
Park                              318
Beaches                             6
Points of Interest & Landmarks      3
Nature & Wildlife Areas             2
Religious Sites                     2
Waterfalls                          2
National Parks                      2
Volcanos                            2
Ancient Ruins                       2
Water Parks                         1
Neighborhoods                       1
Spas                                1
Speciality Museums                  1
Caverns & Caves                     1
Flea & Street Markets               1
Ballets                             1
Name: count, dtype: int64

Missing/Other types: 0


In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Fill missing cities only for vectorization
item_full["city_from_name"] = item_full["city_from_name"].fillna("Unknown")

# TF-IDF for attraction type
type_vectorizer = TfidfVectorizer()
type_matrix = type_vectorizer.fit_transform(
    item_full["AttractionType_clean"]
)

# TF-IDF for attraction city
city_vectorizer = TfidfVectorizer()
city_matrix = city_vectorizer.fit_transform(
    item_full["city_from_name"]
)

# Similarity matrices
type_similarity = cosine_similarity(type_matrix)
city_similarity = cosine_similarity(city_matrix)

# Notebook 4 weighting
content_similarity = (
    0.3 * type_similarity +
    0.7 * city_similarity
)

print("Content-based similarity created.")
print("Type Similarity Shape:", type_similarity.shape)
print("City Similarity Shape:", city_similarity.shape)
print("Combined Similarity Shape:", content_similarity.shape)

Content-based similarity created.
Type Similarity Shape: (1698, 1698)
City Similarity Shape: (1698, 1698)
Combined Similarity Shape: (1698, 1698)


In [41]:
# Prepare popularity scores for the full attraction catalog

popularity_map = attraction_popularity.set_index(
    "AttractionId"
)["popularity_score"].to_dict()

print("Popularity map created.")
print("Attractions with popularity scores:", len(popularity_map))

Popularity map created.
Attractions with popularity scores: 30


In [42]:
# Create normalized popularity scores for recommendation

popularity_scores = attraction_popularity[
    ["AttractionId", "popularity_score"]
].copy()

# Normalize popularity to 0-1
min_pop = popularity_scores["popularity_score"].min()
max_pop = popularity_scores["popularity_score"].max()

popularity_scores["popularity_normalized"] = (
    (popularity_scores["popularity_score"] - min_pop)
    / (max_pop - min_pop)
)

popularity_map = popularity_scores.set_index(
    "AttractionId"
)["popularity_normalized"].to_dict()

print("Normalized popularity scores created.")
print("Popularity range:",
      round(popularity_scores["popularity_normalized"].min(), 3),
      "to",
      round(popularity_scores["popularity_normalized"].max(), 3))

Normalized popularity scores created.
Popularity range: 0.0 to 1.0


In [43]:
# Calculate user history strength

user_history = df.groupby("UserId").agg(
    interaction_count=("TransactionId", "count"),
    unique_attractions=("AttractionId", "nunique")
).reset_index()

user_history["history_strength"] = (
    user_history["interaction_count"] +
    user_history["unique_attractions"]
)

print("User behavior signal created.")
print("\nHistory Strength Summary:")
print(user_history["history_strength"].describe())

User behavior signal created.

History Strength Summary:
count    33530.000000
mean         2.928870
std          1.960839
min          2.000000
25%          2.000000
50%          2.000000
75%          4.000000
max         66.000000
Name: history_strength, dtype: float64


In [44]:
# Assign recommendation weights based on user history

def get_recommendation_weights(history_count):
    if history_count <= 2:
        return 0.20, 0.30, 0.50   # CF, Content, Popularity
    elif history_count <= 5:
        return 0.35, 0.40, 0.25
    else:
        return 0.45, 0.40, 0.15

user_history[
    ["cf_weight", "content_weight", "popularity_weight"]
] = user_history["interaction_count"].apply(
    lambda x: pd.Series(get_recommendation_weights(x))
)

print("Recommendation weights assigned.")
print(
    user_history[
        ["interaction_count",
         "cf_weight",
         "content_weight",
         "popularity_weight"]
    ].drop_duplicates().sort_values("interaction_count").head(10)
)

Recommendation weights assigned.
     interaction_count  cf_weight  content_weight  popularity_weight
2                    1       0.20             0.3               0.50
21                   2       0.20             0.3               0.50
0                    3       0.35             0.4               0.25
139                  4       0.35             0.4               0.25
119                  5       0.35             0.4               0.25
77                   6       0.45             0.4               0.15
95                   7       0.45             0.4               0.15
189                  8       0.45             0.4               0.15
811                  9       0.45             0.4               0.15
1                   10       0.45             0.4               0.15


In [45]:
# Assign recommendation weights based on user history

def get_recommendation_weights(history_count):
    if history_count <= 2:
        return 0.20, 0.30, 0.50   # CF, Content, Popularity
    elif history_count <= 5:
        return 0.35, 0.40, 0.25
    else:
        return 0.45, 0.40, 0.15

user_history[
    ["cf_weight", "content_weight", "popularity_weight"]
] = user_history["interaction_count"].apply(
    lambda x: pd.Series(get_recommendation_weights(x))
)

print("Recommendation weights assigned.")
print(
    user_history[
        ["interaction_count",
         "cf_weight",
         "content_weight",
         "popularity_weight"]
    ].drop_duplicates().sort_values("interaction_count").head(10)
)

Recommendation weights assigned.
     interaction_count  cf_weight  content_weight  popularity_weight
2                    1       0.20             0.3               0.50
21                   2       0.20             0.3               0.50
0                    3       0.35             0.4               0.25
139                  4       0.35             0.4               0.25
119                  5       0.35             0.4               0.25
77                   6       0.45             0.4               0.15
95                   7       0.45             0.4               0.15
189                  8       0.45             0.4               0.15
811                  9       0.45             0.4               0.15
1                   10       0.45             0.4               0.15


In [46]:
# Create lookup maps for user recommendation weights

user_weight_map = user_history.set_index("UserId")[
    ["cf_weight", "content_weight", "popularity_weight"]
].to_dict("index")

print("User weight map created.")
print("Users covered:", len(user_weight_map))

User weight map created.
Users covered: 33530


In [47]:
def advanced_recommendation(user_id, top_n=5):
    if user_id not in user_weight_map:
        return []

    # User-specific weights
    weights = user_weight_map[user_id]
    cf_weight = weights["cf_weight"]
    content_weight = weights["content_weight"]
    popularity_weight = weights["popularity_weight"]

    # User history
    user_history_df = df[df["UserId"] == user_id]
    visited_attractions = set(user_history_df["AttractionId"])

    # Initialize scores for full catalog
    content_scores = {}
    collaborative_scores = {}
    popularity_scores_dict = {}

    # Score candidates using user's previous attractions
    for _, row in user_history_df.iterrows():
        attraction_id = row["AttractionId"]
        rating = row["Rating"]

        # Content-based scores — full 1,698 catalog
        if attraction_id in content_similarity_df.index:
            similarities = content_similarity_df.loc[attraction_id]

            for candidate_id, similarity in similarities.items():
                if candidate_id not in visited_attractions:
                    content_scores[candidate_id] = (
                        content_scores.get(candidate_id, 0)
                        + rating * similarity
                    )

        # Collaborative scores — only 30 interacted attractions
        if attraction_id in collaborative_similarity_df.index:
            similarities = collaborative_similarity_df.loc[attraction_id]

            for candidate_id, similarity in similarities.items():
                if candidate_id not in visited_attractions:
                    collaborative_scores[candidate_id] = (
                        collaborative_scores.get(candidate_id, 0)
                        + rating * similarity
                    )

    # Popularity scores
    for candidate_id, score in popularity_map.items():
        if candidate_id not in visited_attractions:
            popularity_scores_dict[candidate_id] = score

    # Normalize each signal
    def normalize_scores(score_dict):
        if not score_dict:
            return {}

        max_score = max(score_dict.values())

        if max_score == 0:
            return {k: 0 for k in score_dict}

        return {
            k: v / max_score
            for k, v in score_dict.items()
        }

    content_scores = normalize_scores(content_scores)
    collaborative_scores = normalize_scores(collaborative_scores)
    popularity_scores_dict = normalize_scores(popularity_scores_dict)

    # Combine all signals
    all_candidates = (
        set(content_scores)
        | set(collaborative_scores)
        | set(popularity_scores_dict)
    )

    final_scores = {}

    for candidate_id in all_candidates:
        final_scores[candidate_id] = (
            cf_weight * collaborative_scores.get(candidate_id, 0)
            + content_weight * content_scores.get(candidate_id, 0)
            + popularity_weight * popularity_scores_dict.get(candidate_id, 0)
        )

    # Rank recommendations
    recommendations = sorted(
        final_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_n]

    # Map IDs to attraction names
    attraction_name_map = item_full.set_index(
        "AttractionId"
    )["Attraction"].to_dict()

    return [
        {
            "AttractionId": attraction_id,
            "Attraction": attraction_name_map.get(
                attraction_id, "Unknown"
            ),
            "Score": round(score, 4)
        }
        for attraction_id, score in recommendations
    ]

print("Advanced recommendation function created successfully.")

Advanced recommendation function created successfully.


In [48]:
# Test the advanced recommender with the most active user

test_user = df["UserId"].value_counts().index[0]

print("Test User:", test_user)
print("Interactions:", df[df["UserId"] == test_user].shape[0])

recommendations = advanced_recommendation(
    user_id=test_user,
    top_n=5
)

recommendations

Test User: 60799
Interactions: 59


NameError: name 'content_similarity_df' is not defined

In [49]:
print("Advanced Recommendations for User", test_user)

for i, rec in enumerate(recommendations, 1):
    print(
        f"{i}. {rec['Attraction']} "
        f"(ID: {rec['AttractionId']}, Score: {rec['Score']})"
    )

Advanced Recommendations for User 60799


NameError: name 'recommendations' is not defined

In [ ]:
print("Number of recommendations:", len(recommendations))
print("User visited attractions:", df[df["UserId"] == test_user]["AttractionId"].nunique())
print("Total catalog attractions:", item_full["AttractionId"].nunique())
print("Content score candidates:", len([
    x for x in content_similarity_df.columns
    if x not in set(df[df["UserId"] == test_user]["AttractionId"])
]))

In [50]:
# Recreate Notebook 4's final content-based similarity

item_full["city_from_name"] = item_full["Attraction"].apply(
    lambda x: x.split(" - ")[-1].strip()
    if isinstance(x, str) and " - " in x
    else "Unknown"
)

item_full["city_from_name"] = (
    item_full["city_from_name"]
    .replace({"Unknown": None, "-": None})
    .str.strip()
    .replace("", None)
)

type_dummies = pd.get_dummies(item_full["AttractionType_clean"])
city_dummies = pd.get_dummies(item_full["city_from_name"])

type_sim = cosine_similarity(type_dummies)
city_sim = cosine_similarity(city_dummies)

combined_sim = 0.3 * type_sim + 0.7 * city_sim

content_similarity_df = pd.DataFrame(
    combined_sim,
    index=item_full["AttractionId"],
    columns=item_full["AttractionId"]
)

print("Content-based similarity recreated.")
print("Shape:", content_similarity_df.shape)

Content-based similarity recreated.
Shape: (1698, 1698)


In [51]:
# Generate advanced recommendations for the test user

recommendations = advanced_recommendation(
    user_id=test_user,
    top_n=5
)

print("Number of recommendations:", len(recommendations))
print("\nAdvanced Recommendations:")

for i, rec in enumerate(recommendations, 1):
    print(
        f"{i}. {rec['Attraction']} "
        f"(ID: {rec['AttractionId']}, Score: {rec['Score']})"
    )

Number of recommendations: 5

Advanced Recommendations:
1. Mount Semeru Volcano (ID: 947, Score: 0.5619)
2. Uluwatu Temple (ID: 824, Score: 0.5328)
3. Tanah Lot Temple (ID: 737, Score: 0.5255)
4. Tegenungan Waterfall (ID: 749, Score: 0.4327)
5. Waterbom Bali (ID: 841, Score: 0.4167)


In [52]:
# Validate advanced recommendations

recommended_ids = [rec["AttractionId"] for rec in recommendations]
visited_ids = set(df[df["UserId"] == test_user]["AttractionId"])

already_visited = set(recommended_ids) & visited_ids
unique_recommendations = len(set(recommended_ids))

print("Validation Results")
print("------------------")
print("Recommendations generated:", len(recommendations))
print("Already visited attractions:", len(already_visited))
print("Unique recommendations:", unique_recommendations)
print("No repeated visited attractions:", len(already_visited) == 0)

Validation Results
------------------
Recommendations generated: 5
Already visited attractions: 0
Unique recommendations: 5
No repeated visited attractions: True


In [55]:
# Compare Notebook 4 hybrid recommendations with Advanced recommendations

baseline_recommendations = hybrid_recommendation(
    user_id=test_user,
    top_n=5,
    alpha=0.5
)

print("Notebook 4 - Hybrid Recommendations")
print("------------------------------------")

for i, rec in enumerate(baseline_recommendations, 1):
    print(f"{i}. {rec}")

print("\nNotebook 6 - Advanced Recommendations")
print("-------------------------------------")

for i, rec in enumerate(recommendations, 1):
    print(
        f"{i}. {rec['Attraction']} "
        f"(ID: {rec['AttractionId']}, Score: {rec['Score']})"
    )

NameError: name 'hybrid_recommendation' is not defined

In [56]:
# Recreate Notebook 4 hybrid recommender for baseline comparison

def baseline_hybrid_recommendation(user_id, top_n=5, alpha=0.5):
    user_history_df = df[df["UserId"] == user_id]
    visited_attractions = set(user_history_df["AttractionId"])

    content_scores = {}
    collaborative_scores = {}

    for _, row in user_history_df.iterrows():
        attraction_id = row["AttractionId"]
        rating = row["Rating"]

        # Content-based signal
        if attraction_id in content_similarity_df.index:
            similarities = content_similarity_df.loc[attraction_id]

            for candidate_id, similarity in similarities.items():
                if candidate_id not in visited_attractions:
                    content_scores[candidate_id] = (
                        content_scores.get(candidate_id, 0)
                        + rating * similarity
                    )

        # Collaborative filtering signal
        if attraction_id in collaborative_similarity_df.index:
            similarities = collaborative_similarity_df.loc[attraction_id]

            for candidate_id, similarity in similarities.items():
                if candidate_id not in visited_attractions:
                    collaborative_scores[candidate_id] = (
                        collaborative_scores.get(candidate_id, 0)
                        + rating * similarity
                    )

    # Normalize each signal
    def normalize_scores(score_dict):
        if not score_dict:
            return {}

        max_score = max(score_dict.values())

        if max_score == 0:
            return {k: 0 for k in score_dict}

        return {k: v / max_score for k, v in score_dict.items()}

    content_scores = normalize_scores(content_scores)
    collaborative_scores = normalize_scores(collaborative_scores)

    # Combine CF + Content-Based
    all_candidates = set(content_scores) | set(collaborative_scores)

    hybrid_scores = {}

    for candidate_id in all_candidates:
        hybrid_scores[candidate_id] = (
            alpha * collaborative_scores.get(candidate_id, 0)
            + (1 - alpha) * content_scores.get(candidate_id, 0)
        )

    recommendations = sorted(
        hybrid_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_n]

    attraction_name_map = item_full.set_index(
        "AttractionId"
    )["Attraction"].to_dict()

    return [
        {
            "AttractionId": attraction_id,
            "Attraction": attraction_name_map.get(
                attraction_id, "Unknown"
            ),
            "Score": round(score, 4)
        }
        for attraction_id, score in recommendations
    ]

print("Notebook 4 baseline hybrid function recreated successfully.")

Notebook 4 baseline hybrid function recreated successfully.


In [57]:
# Compare Notebook 4 baseline with Notebook 6 advanced recommender

baseline_recommendations = baseline_hybrid_recommendation(
    user_id=test_user,
    top_n=5,
    alpha=0.5
)

print("Notebook 4 - Baseline Hybrid")
print("----------------------------")

for i, rec in enumerate(baseline_recommendations, 1):
    print(
        f"{i}. {rec['Attraction']} "
        f"(ID: {rec['AttractionId']}, Score: {rec['Score']})"
    )

print("\nNotebook 6 - Advanced Recommendation")
print("------------------------------------")

for i, rec in enumerate(recommendations, 1):
    print(
        f"{i}. {rec['Attraction']} "
        f"(ID: {rec['AttractionId']}, Score: {rec['Score']})"
    )

Notebook 4 - Baseline Hybrid
----------------------------
1. Mount Semeru Volcano (ID: 947, Score: 0.6324)
2. Uluwatu Temple (ID: 824, Score: 0.5)
3. Tanah Lot Temple (ID: 737, Score: 0.4932)
4. Tegenungan Waterfall (ID: 749, Score: 0.4248)
5. Bromo Tengger Semeru National Park (ID: 888, Score: 0.3467)

Notebook 6 - Advanced Recommendation
------------------------------------
1. Mount Semeru Volcano (ID: 947, Score: 0.5619)
2. Uluwatu Temple (ID: 824, Score: 0.5328)
3. Tanah Lot Temple (ID: 737, Score: 0.5255)
4. Tegenungan Waterfall (ID: 749, Score: 0.4327)
5. Waterbom Bali (ID: 841, Score: 0.4167)


In [58]:
# Measure recommendation overlap

baseline_ids = {
    rec["AttractionId"] for rec in baseline_recommendations
}

advanced_ids = {
    rec["AttractionId"] for rec in recommendations
}

common_ids = baseline_ids & advanced_ids

print("Recommendation Comparison")
print("-------------------------")
print("Baseline recommendations:", len(baseline_ids))
print("Advanced recommendations:", len(advanced_ids))
print("Common recommendations:", len(common_ids))
print("New recommendations in Advanced:", len(advanced_ids - baseline_ids))
print("Removed from baseline:", len(baseline_ids - advanced_ids))

Recommendation Comparison
-------------------------
Baseline recommendations: 5
Advanced recommendations: 5
Common recommendations: 4
New recommendations in Advanced: 1
Removed from baseline: 1


In [59]:
# Select users with different interaction-history levels

user_interaction_counts = df.groupby("UserId").size()

limited_history_user = user_interaction_counts[
    user_interaction_counts <= 2
].index[0]

medium_history_user = user_interaction_counts[
    (user_interaction_counts >= 3) &
    (user_interaction_counts <= 5)
].index[0]

rich_history_user = user_interaction_counts[
    user_interaction_counts > 5
].index[0]

print("Test Users")
print("----------")
print("Limited history:", limited_history_user,
      "| Interactions:", user_interaction_counts[limited_history_user])
print("Medium history:", medium_history_user,
      "| Interactions:", user_interaction_counts[medium_history_user])
print("Rich history:", rich_history_user,
      "| Interactions:", user_interaction_counts[rich_history_user])

Test Users
----------
Limited history: 20 | Interactions: 1
Medium history: 14 | Interactions: 3
Rich history: 16 | Interactions: 10


In [60]:
# Test advanced recommender across different user-history levels

test_users = {
    "Limited History": limited_history_user,
    "Medium History": medium_history_user,
    "Rich History": rich_history_user
}

for user_type, user_id in test_users.items():
    print("\n" + "=" * 60)
    print(f"{user_type} User: {user_id}")
    print("=" * 60)

    recommendations_test = advanced_recommendation(
        user_id=user_id,
        top_n=5
    )

    for i, rec in enumerate(recommendations_test, 1):
        print(
            f"{i}. {rec['Attraction']} "
            f"(ID: {rec['AttractionId']}, Score: {rec['Score']})"
        )


Limited History User: 20
1. Sacred Monkey Forest Sanctuary (ID: 640, Score: 0.7)
2. Tegalalang Rice Terrace (ID: 748, Score: 0.3667)
3. Uluwatu Temple (ID: 824, Score: 0.2847)
4. Tanah Lot Temple (ID: 737, Score: 0.2713)
5. Kuta Beach - Bali (ID: 369, Score: 0.2614)

Medium History User: 14
1. Tegenungan Waterfall (ID: 749, Score: 0.4339)
2. Water Castle (Tamansari) (ID: 1280, Score: 0.4318)
3. Malang City Square (ID: 937, Score: 0.4119)
4. Waterbom Bali (ID: 841, Score: 0.3928)
5. Tanah Lot Temple (ID: 737, Score: 0.3447)

Rich History User: 16
1. Tanah Lot Temple (ID: 737, Score: 0.6738)
2. Sanur Beach (ID: 650, Score: 0.634)
3. Seminyak Beach (ID: 673, Score: 0.6282)
4. Kuta Beach - Bali (ID: 369, Score: 0.6004)
5. Tegenungan Waterfall (ID: 749, Score: 0.5425)


In [61]:
# Validate that recommendations do not contain already visited attractions

print("Recommendation Validation")
print("-------------------------")

for user_type, user_id in test_users.items():
    recs = advanced_recommendation(
        user_id=user_id,
        top_n=5
    )

    recommended_ids = {rec["AttractionId"] for rec in recs}
    visited_ids = set(
        df[df["UserId"] == user_id]["AttractionId"]
    )

    already_visited = recommended_ids & visited_ids

    print(
        f"{user_type}: "
        f"{len(recs)} recommendations | "
        f"Already visited: {len(already_visited)} | "
        f"Unique: {len(recommended_ids)}"
    )

Recommendation Validation
-------------------------
Limited History: 5 recommendations | Already visited: 0 | Unique: 5
Medium History: 5 recommendations | Already visited: 0 | Unique: 5
Rich History: 5 recommendations | Already visited: 0 | Unique: 5


In [62]:
# Evaluate recommendation coverage across multiple users

sample_users = df["UserId"].drop_duplicates().sample(
    n=100,
    random_state=42
)

recommended_attractions = set()

for user_id in sample_users:
    recs = advanced_recommendation(
        user_id=user_id,
        top_n=5
    )

    for rec in recs:
        recommended_attractions.add(rec["AttractionId"])

total_catalog_attractions = item_full["AttractionId"].nunique()
coverage_percentage = (
    len(recommended_attractions)
    / total_catalog_attractions
) * 100

print("Recommendation Coverage")
print("-----------------------")
print("Users evaluated:", len(sample_users))
print("Unique attractions recommended:", len(recommended_attractions))
print("Total catalog attractions:", total_catalog_attractions)
print("Catalog coverage:", round(coverage_percentage, 2), "%")

Recommendation Coverage
-----------------------
Users evaluated: 100
Unique attractions recommended: 21
Total catalog attractions: 1698
Catalog coverage: 1.24 %


In [63]:
# Validate recommendation scores and ranking

score_validation_passed = True

for user_type, user_id in test_users.items():
    recs = advanced_recommendation(
        user_id=user_id,
        top_n=5
    )

    scores = [rec["Score"] for rec in recs]

    is_descending = all(
        scores[i] >= scores[i + 1]
        for i in range(len(scores) - 1)
    )

    scores_valid = all(
        0 <= score <= 1
        for score in scores
    )

    print(f"{user_type}:")
    print("  Scores:", scores)
    print("  Descending ranking:", is_descending)
    print("  Scores in [0, 1]:", scores_valid)

    if not is_descending or not scores_valid:
        score_validation_passed = False

print("\nOverall Score Validation:", score_validation_passed)

Limited History:
  Scores: [0.7, 0.3667, 0.2847, 0.2713, 0.2614]
  Descending ranking: True
  Scores in [0, 1]: True
Medium History:
  Scores: [0.4339, 0.4318, 0.4119, 0.3928, 0.3447]
  Descending ranking: True
  Scores in [0, 1]: True
Rich History:
  Scores: [0.6738, 0.634, 0.6282, 0.6004, 0.5425]
  Descending ranking: True
  Scores in [0, 1]: True

Overall Score Validation: True
